# ⚠️ Google Colab Setup (Read First)
To run this lab successfully, you **must** enable the Free GPU.

1. In the menu bar at the top, click **Runtime** > **Change runtime type**.
2. Under "Hardware accelerator", select **T4 GPU** and click **Save**.
3. *(Note: You must be signed in with a free Google account to use Colab)*

> **💡 Solution Available:** Try completing this lab on your own first. When you're done, open **`C3_HOL2_Audio_Generation_and_Adaptation_Solution.ipynb`** from the file browser (left panel) in the same folder to compare your outputs with the expected exemplar.


# C3-HOL2: Transcribe Calls and Generate Visual Summaries for BrightCart

| Field | Detail |
|-------|--------|
| **Course** | Introduction to Multimodal AI with Hugging Face |
| **Module** | M2: Audio, Generation, and Adaptation Strategies |
| **Complexity** | Medium |
| **Duration** | 18 minutes |
| **Environment** | HF Inference API (Whisper) + Google Colab free T4 (Diffusers) |

---

## Learning Objective

**LO2 (Apply/Understand):** Build a pipeline that transcribes audio with Whisper and generates images with Diffusers, and describe how LoRA fine-tuning and multimodal RAG extend VLM capabilities.

---

## Scenario

It is **Tuesday at BrightCart**. The customer success team wants two things:

1. **Automated transcription** of customer support calls
2. **Visual summary images** for internal reports

You build a pipeline combining **Whisper** for audio and **Diffusers** for image generation, then evaluate when **fine-tuning vs. retrieval** is the right adaptation strategy.

---

## Prerequisites

- Google account (for Colab GPU)
- Free Hugging Face account with API token

---

## Setup

Run the cell below to install all required packages.

In [ ]:
# Setup — install required packages
!pip install -q huggingface_hub diffusers transformers torch accelerate
# Imports
from huggingface_hub import InferenceClient
import torch
import requests
import warnings
warnings.filterwarnings('ignore')

# Verify GPU (needed for Diffusers)
if torch.cuda.is_available():
    print(f"✅ GPU available: {torch.cuda.get_device_name(0)}")
else:
    print("⚠️  No GPU — Diffusers image generation will be slow.")
    print("   For Colab: Runtime → Change runtime type → T4 GPU")

print("\n✅ Setup complete!")

In [ ]:
# Authenticate with HF
# 👇 Paste your HF token below (get one at https://huggingface.co/settings/tokens)
HF_TOKEN = "paste-your-token-here"
client = InferenceClient(token=HF_TOKEN)
print("✅ Authenticated!")

---

## Task 1: Transcribe a Customer Call with Whisper (4 min)

Use the HF Inference API to transcribe a short customer support call with Whisper.

In [ ]:
# Download a sample audio file
audio_url = "https://huggingface.co/datasets/Narsil/asr_dummy/resolve/main/1.flac"
audio_path = "brightcart_call.flac"

response = requests.get(audio_url)
with open(audio_path, "wb") as f:
    f.write(response.content)

print(f"✅ Audio file downloaded: {audio_path}")
print(f"   Size: {len(response.content) / 1024:.1f} KB")

In [ ]:
# TODO: Transcribe the audio using Whisper via HF Inference API
result = client.automatic_speech_recognition(audio_path)

print("=== Whisper Transcription ===")
print(f"  Text: {result.text}")

# Assess accuracy
print(f"\n📊 Transcription Assessment:")
print(f"  Word count: {len(result.text.split())}")
print(f"\n💡 Note any domain-specific terms that Whisper might have gotten wrong.")
print(f"   Financial/product terminology is often challenging for general ASR models.")

#### ✅ Verification
You should see a text transcription of the audio file. Assess:
- Is the transcription accurate?
- Would any domain-specific terms (product names, financial terms) be misrecognized?

---

## Task 2: Generate a Visual Summary with Diffusers (5 min)

Load a Stable Diffusion pipeline on Colab's T4 GPU. Generate a visual summary from a text prompt based on the call context.

In [ ]:
# TODO: Load Stable Diffusion and generate an image
from diffusers import StableDiffusionPipeline

# Load the pipeline (use float16 for GPU efficiency)
print("🔄 Loading Stable Diffusion pipeline... (this may take a minute)")

if torch.cuda.is_available():
    pipe = StableDiffusionPipeline.from_pretrained(
        "stabilityai/stable-diffusion-2-1",
        torch_dtype=torch.float16
    ).to("cuda")
else:
    pipe = StableDiffusionPipeline.from_pretrained(
        "stabilityai/stable-diffusion-2-1"
    )

print("✅ Stable Diffusion pipeline loaded!")

In [ ]:
# TODO: Generate a visual summary image
# Experiment with prompt specificity and negative prompts

prompt = "Professional product catalog layout for home goods, clean design, white background, high quality photography, commercial style"
negative_prompt = "blurry, low quality, text, watermark, distorted"

print(f"📝 Prompt: {prompt}")
print(f"🚫 Negative: {negative_prompt}")
print(f"\n🎨 Generating image...")

image = pipe(
    prompt=prompt,
    negative_prompt=negative_prompt,
    num_inference_steps=30,
    guidance_scale=7.5
).images[0]

image.save("visual_summary.png")
print(f"✅ Image saved: visual_summary.png")
print(f"   Size: {image.size}")

# Display the image
image

In [ ]:
# TODO: Experiment with a second prompt — more specific
prompt_v2 = "Infographic showing customer satisfaction metrics for retail, bar charts, professional design, corporate blue and green color scheme"

print(f"📝 Prompt v2: {prompt_v2}")
print(f"🎨 Generating...")

image_v2 = pipe(
    prompt=prompt_v2,
    negative_prompt=negative_prompt,
    num_inference_steps=30
).images[0]

image_v2.save("visual_summary_v2.png")
print(f"✅ Image v2 saved!")
image_v2

#### ✅ Verification
Compare the two generated images:
- Which prompt produced a more useful visual summary?
- How does prompt specificity affect output quality?
- What are the limitations of generated images for business reports?

---

## Task 3: Review a Pre-Built LoRA Fine-Tuning Notebook (4 min)

Below is a **read-only reference** showing a LoRA fine-tuning workflow for a VLM. Read through it and answer the comprehension questions.

> ⚠️ **Do not execute this code** — it's for understanding the pattern only. Full LoRA training requires more time and compute than this lab allows.

In [ ]:
# ═══════════════════════════════════════════════════════════
# 📖 READ-ONLY REFERENCE: LoRA Fine-Tuning Workflow
# DO NOT EXECUTE — for understanding the pattern only
# ═══════════════════════════════════════════════════════════

# Step 1: Load the base model
# from transformers import AutoModelForCausalLM
# model = AutoModelForCausalLM.from_pretrained("base-model-id")

# Step 2: Configure LoRA adapter
# from peft import LoraConfig, get_peft_model
# lora_config = LoraConfig(
#     r=16,                    # Rank — controls adapter capacity
#     lora_alpha=32,           # Scaling factor
#     target_modules=["q_proj", "v_proj"],  # Which layers to adapt
#     lora_dropout=0.05,
# )
# model = get_peft_model(model, lora_config)

# Step 3: Train on domain data
# trainer = Trainer(model=model, args=training_args, ...)
# trainer.train()

# Step 4: Merge and save
# model = model.merge_and_unload()
# model.push_to_hub("my-lora-adapted-model")

print("📖 This is a READ-ONLY reference notebook.")
print("   Review the 4 steps above and answer the questions below.")
print(f"\n   Key LoRA concepts:")
print(f"   - r (rank): Controls adapter capacity (more rank = more parameters)")
print(f"   - target_modules: Which attention layers to adapt")
print(f"   - merge_and_unload(): Combines adapter weights with base model")
print(f"   - Only ~0.1-1% of parameters are trained (very efficient)")

### LoRA Comprehension Questions

1. *TODO: What does the `r` (rank) parameter control? What happens if you increase it?*
2. *TODO: Why does LoRA only target specific modules (q_proj, v_proj)? What would happen if you targeted all layers?*
3. *TODO: What does `merge_and_unload()` do? Why is it useful for deployment?*
4. *TODO: Approximately what percentage of parameters are trained with LoRA vs full fine-tuning?*

---

## Task 4: Review a Multimodal RAG Example (3 min)

Below is a **read-only reference** showing a multimodal RAG workflow. The key difference: **RAG changes what the model sees, not what the model knows.**

In [ ]:
# ═══════════════════════════════════════════════════════════
# 📖 READ-ONLY REFERENCE: Multimodal RAG Workflow
# DO NOT EXECUTE — for understanding the pattern only
# ═══════════════════════════════════════════════════════════

# Step 1: Index product catalog images
# embeddings = clip_model.encode_images(catalog_images)
# vector_store.add(embeddings, metadata)

# Step 2: User asks a question about a product
# query = "What's the thread count of the blue duvet?"

# Step 3: Retrieve relevant catalog pages
# query_embedding = clip_model.encode_text(query)
# relevant_pages = vector_store.search(query_embedding, top_k=3)

# Step 4: Feed retrieved context to VLM
# grounded_answer = vlm.generate(
#     images=relevant_pages,
#     prompt=f"Based on these catalog pages, {query}"
# )

# Step 5: Compare grounded vs ungrounded answers
# ungrounded = vlm.generate(prompt=query)  # No context
# The grounded answer uses actual product data
# The ungrounded answer may hallucinate

print("📖 This is a READ-ONLY reference notebook.")
print(f"\n   Key RAG concepts:")
print(f"   - RAG retrieves relevant documents as context")
print(f"   - The model's weights are NOT changed")
print(f"   - Grounded answers use real data → less hallucination")
print(f"   - Works with any base model (no training required)")

### RAG vs LoRA Key Difference

- **LoRA** changes **what the model knows** (adapts its weights)
- **RAG** changes **what the model sees** (provides relevant context)

---

## Task 5: Document the Adaptation Strategy Recommendation (2 min)

### 📋 Deliverable D2: Adaptation Strategy Recommendation

Given BrightCart's 30% caption error rate on home goods products:

| Question | Your Answer |
|----------|-------------|
| **When would you recommend LoRA fine-tuning?** | *TODO* |
| **When would you recommend RAG?** | *TODO* |
| **Which factors drive the decision?** | *TODO* |
| **For BrightCart's specific case, which do you recommend?** | *TODO: LoRA / RAG / Both?* |
| **Why?** | *TODO: Explain your reasoning* |

---

## Key Takeaways

In this lab, you:
1. **Transcribed audio with Whisper** via the HF Inference API
2. **Generated images with Stable Diffusion** using the Diffusers library
3. **Reviewed LoRA fine-tuning** — adapting model weights for domain-specific tasks
4. **Reviewed multimodal RAG** — grounding model outputs with retrieved context
5. **Wrote an adaptation strategy** for BrightCart's caption quality problem

**Key insight:** LoRA changes what the model *knows*; RAG changes what the model *sees*. The right choice depends on your data, compute budget, and update frequency.

**Next up:** Wednesday (C3-HOL3) — you'll build an agent that automates BrightCart's catalog workflow.

---
*© BrightCart — Course 3, Day 2 (Tuesday)*